In [1]:
!pip install transformers datasets accelerate pandas

# Step 1: Load and prepare emotion dataset

In [2]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("emotion_predictions_full.csv")

label_map = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

df["emotion"] = df["predicted_label"].map(label_map).fillna("neutral")

def create_output(row):
    emotion = row["emotion"]

    if emotion == "anger":
        return "Reason: The user expresses frustration or irritation. Feedback: Acknowledge the concern, stay calm, and resolve the root issue with clear next steps."
    elif emotion == "joy":
        return "Reason: The user expresses happiness and satisfaction. Feedback: Reinforce what worked well and maintain this positive experience."
    elif emotion == "sadness":
        return "Reason: The user expresses disappointment or sadness. Feedback: Show empathy, offer support, and suggest practical improvements."
    elif emotion == "love":
        return "Reason: The user expresses affection or strong positive feelings. Feedback: Preserve trust and continue nurturing this connection."
    elif emotion == "fear":
        return "Reason: The user expresses fear or uncertainty. Feedback: Provide reassurance, explain clearly, and reduce ambiguity."
    elif emotion == "surprise":
        return "Reason: The user expresses an unexpected reaction. Feedback: Add context and improve clarity to prevent confusion."
    else:
        return "Reason: The user shows a neutral tone. Feedback: Increase engagement with clearer value and personalization."

df["instruction"] = "Explain the emotion and give improvement feedback"
df["input"] = "Text: " + df["text"].astype(str) + " | Emotion: " + df["emotion"].astype(str)
df["output"] = df.apply(create_output, axis=1)

dataset = Dataset.from_pandas(df[["instruction", "input", "output"]])
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

df[["instruction", "input", "output"]].sample(5, random_state=42)

,instruction,input,output
8756,Explain the emotion and give improvement feedback,Text: ive made it through a week i just feel b...,Reason: The user expresses disappointment or s...
4660,Explain the emotion and give improvement feedback,Text: i feel this strategy is worthwhile | Emo...,Reason: The user expresses happiness and satis...
6095,Explain the emotion and give improvement feedback,Text: i feel so worthless and weak what does h...,Reason: The user expresses disappointment or s...
304,Explain the emotion and give improvement feedback,Text: i feel clever nov | Emotion: joy,Reason: The user expresses happiness and satis...
8241,Explain the emotion and give improvement feedback,Text: im moved in ive been feeling kind of glo...,Reason: The user expresses disappointment or s...


# Step 2: Load pretrained GPT-2 tokenizer and model

In [3]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# Step 3: Tokenize dataset

In [4]:
BEHAVIOR_PROMPT = (
    "You are an emotion reasoning assistant. Always identify emotional signals from the input text, "
    "explain the likely reason in 1-2 sentences, and provide practical, respectful improvement feedback. "
    "Do not be rude, judgmental, or overly verbose. Keep responses clear and supportive."
)

def build_prompt(instruction, input_text, output_text):
    return (
        f"### System Behavior:\n{BEHAVIOR_PROMPT}\n\n"
        + f"### Instruction:\n{instruction}\n\n"
        + f"### Input:\n{input_text}\n\n"
        + f"### Response:\n{output_text}"
    )

def tokenize_function(examples):
    prompts = [
        build_prompt(inst, inp, out)
        for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"])
    ]

    tokens = tokenizer(
        prompts,
        padding="max_length",
        truncation=True,
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "input", "output"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "input", "output"])

Map:   0%|          | 0/14400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

# Step 4: Create data collator

In [5]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [21]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,  # LoRA attention dimension
    lora_alpha=16,  # Alpha parameter for LoRA scaling
    target_modules=["c_attn", "c_proj", "c_fc"],  # Modules to apply LoRA to
    lora_dropout=0.05,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA layers
    task_type="CAUSAL_LM"  # Task type for causal language modeling
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 1,179,648 || all params: 125,619,456 || trainable%: 0.9391


# Step 5 and 6: Training arguments and Trainer

In [22]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./fine-tuned-gpt2",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    save_steps=1000,
    save_total_limit=2,
    logging_steps=100,
    eval_strategy="epoch",
    eval_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)

# Step 7 and 8: Train and save model

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.581681,0.637045
2,0.556369,0.641679


TrainOutput(global_step=1800, training_loss=0.5438772243923611, metrics={'train_runtime': 446.0767, 'train_samples_per_second': 64.563, 'train_steps_per_second': 4.035, 'total_flos': 1907394556723200.0, 'train_loss': 0.5438772243923611, 'epoch': 2.0})

In [24]:
model.save_pretrained("./fine-tuned-gpt2")
tokenizer.save_pretrained("./fine-tuned-gpt2")

('./fine-tuned-gpt2/tokenizer_config.json', './fine-tuned-gpt2/tokenizer.json')

# Step 9: Inference with fine-tuned model

### Test with a single custom input

In [31]:
import os

# Define the directory to save the model and tokenizer for Streamlit
output_dir = "./fine-tuned-gpt2-streamlit"
os.makedirs(output_dir, exist_ok=True)

# Save only the LoRA adapters
# The 'model' object is already a PeftModel instance
model.save_pretrained(output_dir)

# Save the tokenizer
tokenizer.save_pretrained(output_dir)

print(f"Model adapters and tokenizer saved to: {output_dir}")

Model adapters and tokenizer saved to: ./fine-tuned-gpt2-streamlit
